In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import os

ROOT_DIR = "/content/drive/MyDrive/nutrition_rag"

print(os.path.exists(ROOT_DIR))

True


In [4]:
import os

ROOT_DIR = "/content/drive/MyDrive/nutrition_rag"

RAW_DATA_DIR = os.path.join(ROOT_DIR, "data_raw")
PROCESSED_DIR = os.path.join(ROOT_DIR, "data_processed")
VECTOR_DB_DIR = os.path.join(ROOT_DIR, "vector_db")
OUTPUT_DIR = os.path.join(ROOT_DIR, "outputs")

for folder in [
    PROCESSED_DIR,
    VECTOR_DB_DIR,
    OUTPUT_DIR
]:
    os.makedirs(folder, exist_ok=True)

print("Folders ready")

food_pdf = os.path.join(
    RAW_DATA_DIR,
    "food_composition_vn_2007.pdf"
)
nutrition_pdf = os.path.join(
    RAW_DATA_DIR,
    "nutrition_recommendation_vn.pdf"
)
hospital_pdf = os.path.join(
    RAW_DATA_DIR,
    "hospital_diet_guideline.pdf"
)
usage_pdf = os.path.join(
    RAW_DATA_DIR,
    "food_composition_usage.pdf"
)
print(food_pdf, nutrition_pdf, hospital_pdf, usage_pdf)

Folders ready
/content/drive/MyDrive/nutrition_rag/data_raw/food_composition_vn_2007.pdf /content/drive/MyDrive/nutrition_rag/data_raw/nutrition_recommendation_vn.pdf /content/drive/MyDrive/nutrition_rag/data_raw/hospital_diet_guideline.pdf /content/drive/MyDrive/nutrition_rag/data_raw/food_composition_usage.pdf


In [5]:
!pip install -q \
pdfplumber \
pymupdf \
pandas \
openpyxl \
tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 87.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 82.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 124.6 MB/s eta 0:00:00


In [11]:
!pip install -q ftfy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.4 MB/s eta 0:00:00


In [6]:
import pdfplumber

with pdfplumber.open(food_pdf) as pdf:
    print("Pages:", len(pdf.pages))

    first_page = pdf.pages[0]

    text = first_page.extract_text()

    print(text[:2000])

Pages: 567
BỘ Y TẾ
VIỆN DINH DƯỠNG
(cid:94)(cid:92)(cid:93)(cid:91)
BẢNG THÀNH PHẦN
TTHHỰỰCC PPHHẨẨMM VVIIỆỆTT NNAAMM
VIETNAMESE FOOD COMPOSITION TABLE
NHÀ XUẤT BẢN Y HỌC


In [8]:
import pdfplumber
from tqdm import tqdm

pages_data = []

with pdfplumber.open(food_pdf) as pdf:

    for page_idx, page in enumerate(tqdm(pdf.pages)):

        text = page.extract_text()

        if text and len(text.strip()) > 100:

            pages_data.append({
                "page": page_idx + 1,
                "text": text
            })

print("Total pages extracted:", len(pages_data))

100%|██████████| 567/567 [01:28<00:00,  6.43it/s]

Total pages extracted: 553


In [9]:
import json
import os

output_file = os.path.join(
    PROCESSED_DIR,
    "food_composition_pages.json"
)

with open(
    output_file,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        pages_data,
        f,
        ensure_ascii=False,
        indent=2
    )

print(output_file)

/content/drive/MyDrive/nutrition_rag/data_processed/food_composition_pages.json


In [27]:
import re


def extract_food_metadata(text):

    result = {
        "food_name_en": None,
        "food_code": None,
        "energy_kcal": None,
        "protein_g": None,
        "fat_g": None,
        "carb_g": None,
        "fiber_g": None,
        "calcium_mg": None,
        "iron_mg": None
    }

    # -----------------
    # English name
    # -----------------

    m = re.search(
        r"English\):\s*(.+?)\s*M.\s*s",
        text,
        re.DOTALL
    )

    if m:
        result["food_name_en"] = m.group(1).strip()

    # -----------------
    # Food code
    # -----------------

    m = re.search(
        r"M.\s*s.:\s*(\d+)",
        text
    )

    if m:
        result["food_code"] = m.group(1)

    # -----------------
    # Energy
    # -----------------

    m = re.search(
        r"Năng lượng \(Energy\)\s*KCal\s*([\d\.]+)",
        text
    )

    if m:
        result["energy_kcal"] = float(m.group(1))

    # -----------------
    # Protein
    # -----------------

    m = re.search(
        r"Protein\s*g\s*([\d\.]+)",
        text
    )

    if m:
        result["protein_g"] = float(m.group(1))

    # -----------------
    # Fat
    # -----------------

    m = re.search(
        r"Lipid \(Fat\)\s*g\s*([\d\.]+)",
        text
    )

    if m:
        result["fat_g"] = float(m.group(1))

    # -----------------
    # Carb
    # -----------------

    m = re.search(
        r"Glucid \(Carbohydrate\)\s*g\s*([\d\.]+)",
        text
    )

    if m:
        result["carb_g"] = float(m.group(1))

    # -----------------
    # Fiber
    # -----------------

    m = re.search(
        r"Celluloza \(Fiber\)\s*g\s*([\d\.]+)",
        text
    )

    if m:
        result["fiber_g"] = float(m.group(1))

    # -----------------
    # Calcium
    # -----------------

    m = re.search(
        r"Calci \(Calcium\)\s*mg\s*([\d\.]+)",
        text
    )

    if m:
        result["calcium_mg"] = float(m.group(1))

    # -----------------
    # Iron
    # -----------------

    m = re.search(
        r"Sắt \(Iron\)\s*mg\s*([\d\.]+)",
        text
    )

    if m:
        result["iron_mg"] = float(m.group(1))

    return result

In [36]:
food_metadata = []

for record in pages_data:

    metadata = extract_food_metadata(
        record["text"]
    )

    metadata["page"] = record["page"]

    metadata["source"] = "food_composition_vn_2007"

    metadata["raw_text"] = record["text"]

    food_metadata.append(metadata)

print("Total records:", len(food_metadata))

Total records: 553


In [37]:
import pandas as pd

df = pd.DataFrame(food_metadata)

df.head()

,food_name_en,food_code,energy_kcal,protein_g,fat_g,carb_g,fiber_g,calcium_mg,iron_mg,page,source,raw_text
0,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,food_composition_vn_2007,BỘ Y TẾ\nVIỆN DINH DƯỠNG\n(cid:94)(cid:92)(cid...
1,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,food_composition_vn_2007,BỘ Y TẾ\nVIỆN DINH DƯỠNG\n(cid:94)(cid:93)(cid...
2,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,food_composition_vn_2007,MỤC LỤC\nLời mở đầu iii\nGiới thiệu Bảng thành...
3,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4,food_composition_vn_2007,LỜI NÓI ĐẦU\nBảng thành phần thực phẩm (Food C...
4,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5,food_composition_vn_2007,GIỚI THIỆU BẢNG THÀNH PHẦN THỰC PHẨM VIỆT NAM\...


In [38]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 553 entries, 0 to 552
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   food_name_en  526 non-null    object 
 1   food_code     526 non-null    object 
 2   energy_kcal   526 non-null    float64
 3   protein_g     525 non-null    float64
 4   fat_g         451 non-null    float64
 5   carb_g        526 non-null    float64
 6   fiber_g       506 non-null    float64
 7   calcium_mg    479 non-null    float64
 8   iron_mg       427 non-null    float64
 9   page          553 non-null    int64  
 10  source        553 non-null    object 
 11  raw_text      553 non-null    object 
dtypes: float64(7), int64(1), object(4)
memory usage: 52.0+ KB


In [39]:
df[
    [
        "food_name_en",
        "energy_kcal",
        "protein_g",
        "fat_g",
        "carb_g"
    ]
].sample(10)

,food_name_en,energy_kcal,protein_g,fat_g,carb_g
430,"Sea shrimp, sea-water shrimp",82.0,17.6,0.9,0.9
149,"Asparagus, white",14.0,2.2,0.1,1.1
527,"Fish sauce, grade II",21.0,5.2,0.0,0.0
305,"Dog, shoulder",230.0,18.0,17.6,0.0
366,"Pork, head meat, steamed",553.0,16.0,54.3,0.0
344,Chicken gizzard,99.0,21.3,1.3,0.6
82,"Sesame oriental seeds, whole, dried black or w...",568.0,20.1,46.4,17.6
261,"Grape, European, sweet",68.0,0.4,0.2,16.3
459,"Milk condensed, sweetened",336.0,8.1,8.8,56.0
50,Vermicelli from Bermuda tuber,332.0,0.6,0.1,82.2


In [40]:
import json
import os

metadata_file = os.path.join(
    PROCESSED_DIR,
    "food_metadata.json"
)

with open(
    metadata_file,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        food_metadata,
        f,
        ensure_ascii=False,
        indent=2
    )

print(metadata_file)

/content/drive/MyDrive/nutrition_rag/data_processed/food_metadata.json


In [41]:
import pandas as pd

df = pd.DataFrame(food_metadata)

print("Total:", len(df))

print(
    "Valid food records:",
    df["food_name_en"].notna().sum()
)

Total: 553
Valid food records: 526


In [42]:
food_metadata_clean = [
    x
    for x in food_metadata
    if x["food_name_en"] is not None
]

In [46]:
import json
import os

metadata_file = os.path.join(
    PROCESSED_DIR,
    "food_metadata.json"
)

with open(
    metadata_file,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        food_metadata_clean,
        f,
        ensure_ascii=False,
        indent=2
    )

print(metadata_file)
print("Records:", len(food_metadata_clean))

/content/drive/MyDrive/nutrition_rag/data_processed/food_metadata.json
Records: 526


In [51]:
import fitz
from tqdm import tqdm

pages = []

doc = fitz.open(recommendation_pdf)

for page_num in tqdm(range(len(doc))):

    text = doc[page_num].get_text()

    pages.append({
        "page": page_num + 1,
        "text": text
    })

doc.close()

print("Pages:", len(pages))

100%|██████████| 8/8 [00:00<00:00, 97.00it/s]

Pages: 8


In [52]:
recommendation_pages_file = os.path.join(
    PROCESSED_DIR,
    "recommendation_pages.json"
)

with open(
    recommendation_pages_file,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        pages,
        f,
        ensure_ascii=False,
        indent=2
    )

print(recommendation_pages_file)

/content/drive/MyDrive/nutrition_rag/data_processed/recommendation_pages.json


In [61]:
import pdfplumber

with pdfplumber.open(recommendation_pdf) as pdf:

    print("Total pages:", len(pdf.pages))

    for page_idx in [1, 3, 5]:

        tables = pdf.pages[page_idx].extract_tables()

        print("\n" + "=" * 80)
        print("PAGE", page_idx + 1)
        print("TABLES:", len(tables))

        if tables:
            print("ROWS:", len(tables[0]))

Total pages: 8

PAGE 2
TABLES: 1
ROWS: 13

PAGE 4
TABLES: 1
ROWS: 23

PAGE 6
TABLES: 1
ROWS: 25


In [63]:
import pdfplumber

with pdfplumber.open(recommendation_pdf) as pdf:

    mineral_table_p2 = pdf.pages[1].extract_tables()[0]
    mineral_table_p3 = pdf.pages[2].extract_tables()[0]

print("P2 rows:", len(mineral_table_p2))
print("P3 rows:", len(mineral_table_p3))

P2 rows: 13
P3 rows: 19


In [64]:
mineral_table = (
    mineral_table_p2 +
    mineral_table_p3
)

print("Merged rows:", len(mineral_table))

Merged rows: 32


In [73]:
KNOWN_GROUPS = {
    "Trẻ em",
    "Trẻ nhỏ",
    "Nam vị thành niên",
    "Nam trưởng thành",
    "Nữ vị thành niên",
    "Nữ trưởng thành",
    "Phụ nữ mang thai",
    "Bà mẹ cho con bú (trong suốt cả thời kỳ cho bú)"
}

def is_group_row(row):

    if not row[0]:
        return False

    name = row[0].replace("\n", " ").strip()

    return name in KNOWN_GROUPS

In [74]:
for row in mineral_table[:15]:

    print(
        row[0],
        "->",
        is_group_row(row)
    )

Nhóm tuổi, giới -> False
Trẻ em -> True
< 6 tháng -> False
6-11 tháng -> False
Trẻ nhỏ -> True
1-3 tuổi -> False
4-6 tuổi -> False
7-9 tuổi -> False
Nam vị thành niên -> True
10-12 tuổi -> False
13-15 tuổi -> False
16-18 tuổi -> False
Nam trưởng thành -> True
19-49 tuổi -> False
50-60 tuổi -> False


In [75]:
records = []

current_group = None

for row in mineral_table[1:]:

    age = row[0]

    if not age:
        continue

    if is_group_row(row):

        current_group = (
            age
            .replace("\n", " ")
            .strip()
        )

        continue

    records.append({
        "group": current_group,
        "age": age.replace("\n", " ").strip(),

        "calcium_mg": row[1],
        "magnesium_mg": row[2],
        "phosphorus_mg": row[3],
        "selenium_mcg": row[4]
    })

In [76]:
records_ffill = []

current_group = None

last_calcium = None
last_magnesium = None
last_phosphorus = None
last_selenium = None

for row in mineral_table[1:]:

    age = row[0]

    if not age:
        continue

    if is_group_row(row):

        current_group = (
            age
            .replace("\n", " ")
            .strip()
        )

        continue

    calcium = row[1] if row[1] not in [None, ""] else last_calcium
    magnesium = row[2] if row[2] not in [None, ""] else last_magnesium
    phosphorus = row[3] if row[3] not in [None, ""] else last_phosphorus
    selenium = row[4] if row[4] not in [None, ""] else last_selenium

    last_calcium = calcium
    last_magnesium = magnesium
    last_phosphorus = phosphorus
    last_selenium = selenium

    records_ffill.append({
        "group": current_group,
        "age": age.replace("\n", " ").strip(),

        "calcium_mg": calcium,
        "magnesium_mg": magnesium,
        "phosphorus_mg": phosphorus,
        "selenium_mcg": selenium
    })

In [77]:
for r in records_ffill[:15]:
    print(r)

{'group': 'Trẻ em', 'age': '< 6 tháng', 'calcium_mg': '300', 'magnesium_mg': '36', 'phosphorus_mg': '90', 'selenium_mcg': '6'}
{'group': 'Trẻ em', 'age': '6-11 tháng', 'calcium_mg': '400', 'magnesium_mg': '54', 'phosphorus_mg': '275', 'selenium_mcg': '10'}
{'group': 'Trẻ nhỏ', 'age': '1-3 tuổi', 'calcium_mg': '500', 'magnesium_mg': '65', 'phosphorus_mg': '460', 'selenium_mcg': '17'}
{'group': 'Trẻ nhỏ', 'age': '4-6 tuổi', 'calcium_mg': '600', 'magnesium_mg': '76', 'phosphorus_mg': '500', 'selenium_mcg': '22'}
{'group': 'Trẻ nhỏ', 'age': '7-9 tuổi', 'calcium_mg': '700', 'magnesium_mg': '100', 'phosphorus_mg': '500', 'selenium_mcg': '21'}
{'group': 'Nam vị thành niên', 'age': '10-12 tuổi', 'calcium_mg': '1.000', 'magnesium_mg': '155', 'phosphorus_mg': '1.250', 'selenium_mcg': '32'}
{'group': 'Nam vị thành niên', 'age': '13-15 tuổi', 'calcium_mg': '1.000', 'magnesium_mg': '225', 'phosphorus_mg': '1.250', 'selenium_mcg': '32'}
{'group': 'Nam vị thành niên', 'age': '16-18 tuổi', 'calcium_mg

In [78]:
def to_number(value):

    if value is None:
        return None

    value = str(value).strip()

    if value == "":
        return None

    value = value.replace(".", "")
    value = value.replace(",", ".")

    try:
        num = float(value)

        if num.is_integer():
            return int(num)

        return num

    except:
        return value

mineral_records = []

for r in records_ffill:

    mineral_records.append({
        "record_type": "mineral",

        "group": r["group"],
        "age": r["age"],

        "calcium_mg": to_number(r["calcium_mg"]),
        "magnesium_mg": to_number(r["magnesium_mg"]),
        "phosphorus_mg": to_number(r["phosphorus_mg"]),
        "selenium_mcg": to_number(r["selenium_mcg"]),

        "source": "nutrition_recommendation_vn.pdf"
    })

print("Records:", len(mineral_records))

for r in mineral_records[:5]:
    print(r)

Records: 23
{'record_type': 'mineral', 'group': 'Trẻ em', 'age': '< 6 tháng', 'calcium_mg': 300, 'magnesium_mg': 36, 'phosphorus_mg': 90, 'selenium_mcg': 6, 'source': 'nutrition_recommendation_vn.pdf'}
{'record_type': 'mineral', 'group': 'Trẻ em', 'age': '6-11 tháng', 'calcium_mg': 400, 'magnesium_mg': 54, 'phosphorus_mg': 275, 'selenium_mcg': 10, 'source': 'nutrition_recommendation_vn.pdf'}
{'record_type': 'mineral', 'group': 'Trẻ nhỏ', 'age': '1-3 tuổi', 'calcium_mg': 500, 'magnesium_mg': 65, 'phosphorus_mg': 460, 'selenium_mcg': 17, 'source': 'nutrition_recommendation_vn.pdf'}
{'record_type': 'mineral', 'group': 'Trẻ nhỏ', 'age': '4-6 tuổi', 'calcium_mg': 600, 'magnesium_mg': 76, 'phosphorus_mg': 500, 'selenium_mcg': 22, 'source': 'nutrition_recommendation_vn.pdf'}
{'record_type': 'mineral', 'group': 'Trẻ nhỏ', 'age': '7-9 tuổi', 'calcium_mg': 700, 'magnesium_mg': 100, 'phosphorus_mg': 500, 'selenium_mcg': 21, 'source': 'nutrition_recommendation_vn.pdf'}


In [79]:
import json
import os

mineral_file = os.path.join(
    PROCESSED_DIR,
    "recommendation_minerals.json"
)

with open(
    mineral_file,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        mineral_records,
        f,
        ensure_ascii=False,
        indent=2
    )

print(mineral_file)
print("Records:", len(mineral_records))

/content/drive/MyDrive/nutrition_rag/data_processed/recommendation_minerals.json
Records: 23


In [84]:
import pdfplumber

with pdfplumber.open(recommendation_pdf) as pdf:

    for page_idx in [3, 4]:

        tables = pdf.pages[page_idx].extract_tables()

        print(
            f"PAGE {page_idx+1}",
            "TABLES:",
            len(tables)
        )

print(pages[4]["text"])

for row in trace_table_p4[-10:]:
    print(row)

PAGE 4 TABLES: 1
PAGE 5 TABLES: 0
17 
 
8 Không áp dụng cho trẻ bú sữa mẹ đơn thuần 
9 Hấp thu tốt: giá trị sinh học kẽm tốt = 50% (khẩu phần có nhiều protein động vật 
hoặc cá); hấp thu vừa: giá trị sinh học kẽm trung bình = 30% (khẩu phần có vừa phải 
protein động vật hoặc cá; tỷ số phytat-kẽm phân tử là 5:15). Hấp thu kém: giá trị sinh 
học kẽm thấp =15% (khẩu phần ít hoặc không có protein động vật hoặc cá). 

['10-14 tuổi', '120', '28,0', '18,7', '14,0', '4,6', '7,8', '15,5']
['15-18 tuổi', '150', '65,4', '43,6', '32,7', '4,6', '7,8', '15,5']
['Người trưởng thành', None, None, None, None, None, None, None]
['Nam ≥ 19 tuổi', '150', '27,4', '18,3', '13,7', '4,2', '7,0', '14,0']
['Nữ ≥ 19 tuổi', '150', '58,8', '39,2', '29,4', '3,0', '4,9', '9,8']
['Trung niên ≥ 50 tuổi', None, None, None, None, None, None, None]
['Nam', '', '', '', '', '3,0', '4,9', '9,8']
['Nữ', '', '22,6', '15,1', '11,3', '3,0', '4,9', '9,8']
['Phụ nữ có thai', '200', '+30,04', '+20,04', '+15,04', '', '', '']
['Phụ 

In [85]:
for row in trace_table_p4[:20]:
    print(row)

['Nhóm tuổi', 'Iốt\n(µg/ngày)', 'Sắt (mg/ngày) theo\ngiá trị sinh học khẩu\nphần', None, None, 'Kẽm (mg/ngày)', None, None]
[None, None, '5%1', '10%2', '15%3', 'Hấp\nthu tốt', 'Hấp\nthu\nvừa', 'Hấp\nthu\nkém']
['Trẻ em', None, None, None, None, None, None, None]
['0-6 tháng', '90', '0,93', '', '', '1,15', '2,86', '6,57']
['6-11 tháng', '90', '18,6', '12,4', '9,3', '0,8-\n2,58', '4,18', '8,38']
['Trẻ nhỏ', None, None, None, None, None, None, None]
['1-3 tuổi', '90', '11,6', '7,7', '5,8', '2,4', '4,1', '8,4']
['4-6 tuổi', '90', '12,6', '8,4', '6,3', '3,1', '5,1', '10,3']
['7-9 tuổi', '90', '17,8', '11,9', '8,9', '3,3', '5,6', '11,3']
['Nam vị thành niên', None, None, None, None, None, None, None]
['10-14 tuổi', '120', '29,2', '19,5', '14,6', '5,7', '9,7', '19,2']
['15-18 tuổi', '150', '37,6', '25,1', '18,8', '5,7', '9,7', '19,2']
['Nữ vị thành niên', None, None, None, None, None, None, None]
['10-14 tuổi', '120', '28,0', '18,7', '14,0', '4,6', '7,8', '15,5']
['15-18 tuổi', '150', '65,4',

In [89]:
TRACE_GROUPS = {
    "Trẻ em",
    "Trẻ nhỏ",
    "Nam vị thành niên",
    "Nữ vị thành niên",
    "Người trưởng thành",
    "Trung niên ≥ 50 tuổi"
}
def is_trace_group(row):

    if not row[0]:
        return False

    name = row[0].replace("\n", " ").strip()

    return name in TRACE_GROUPS
trace_records = []

current_group = None

for row in trace_table_p4[2:]:

    age = row[0]

    if not age:
        continue

    if is_trace_group(row):

        current_group = (
            age
            .replace("\n", " ")
            .strip()
        )

        continue

    trace_records.append({
        "group": current_group,
        "age": age.replace("\n", " ").strip(),

        "iodine_mcg": row[1],

        "iron_5pct_mg": row[2],
        "iron_10pct_mg": row[3],
        "iron_15pct_mg": row[4],

        "zinc_good_mg": row[5],
        "zinc_medium_mg": row[6],
        "zinc_low_mg": row[7]
    })

print("Records:", len(trace_records))

for r in trace_records[:10]:
    print(r)



Records: 15
{'group': 'Trẻ em', 'age': '0-6 tháng', 'iodine_mcg': '90', 'iron_5pct_mg': '0,93', 'iron_10pct_mg': '', 'iron_15pct_mg': '', 'zinc_good_mg': '1,15', 'zinc_medium_mg': '2,86', 'zinc_low_mg': '6,57'}
{'group': 'Trẻ em', 'age': '6-11 tháng', 'iodine_mcg': '90', 'iron_5pct_mg': '18,6', 'iron_10pct_mg': '12,4', 'iron_15pct_mg': '9,3', 'zinc_good_mg': '0,8-\n2,58', 'zinc_medium_mg': '4,18', 'zinc_low_mg': '8,38'}
{'group': 'Trẻ nhỏ', 'age': '1-3 tuổi', 'iodine_mcg': '90', 'iron_5pct_mg': '11,6', 'iron_10pct_mg': '7,7', 'iron_15pct_mg': '5,8', 'zinc_good_mg': '2,4', 'zinc_medium_mg': '4,1', 'zinc_low_mg': '8,4'}
{'group': 'Trẻ nhỏ', 'age': '4-6 tuổi', 'iodine_mcg': '90', 'iron_5pct_mg': '12,6', 'iron_10pct_mg': '8,4', 'iron_15pct_mg': '6,3', 'zinc_good_mg': '3,1', 'zinc_medium_mg': '5,1', 'zinc_low_mg': '10,3'}
{'group': 'Trẻ nhỏ', 'age': '7-9 tuổi', 'iodine_mcg': '90', 'iron_5pct_mg': '17,8', 'iron_10pct_mg': '11,9', 'iron_15pct_mg': '8,9', 'zinc_good_mg': '3,3', 'zinc_medium_mg

In [92]:
import re

def clean_value(v):

    if v is None:
        return None

    v = str(v).replace("\n", " ").strip()

    if v == "":
        return None

    # giữ nguyên khoảng
    if "-" in v:
        return v

    # giữ nguyên dạng cộng thêm
    if v.startswith("+"):
        return v

    try:
        return float(v.replace(",", "."))
    except:
        return v

trace_records_clean = []

for r in trace_records:

    trace_records_clean.append({
        "record_type": "trace_mineral",

        "group": r["group"],
        "age": r["age"],

        "iodine_mcg": clean_value(r["iodine_mcg"]),

        "iron_5pct_mg": clean_value(r["iron_5pct_mg"]),
        "iron_10pct_mg": clean_value(r["iron_10pct_mg"]),
        "iron_15pct_mg": clean_value(r["iron_15pct_mg"]),

        "zinc_good_mg": clean_value(r["zinc_good_mg"]),
        "zinc_medium_mg": clean_value(r["zinc_medium_mg"]),
        "zinc_low_mg": clean_value(r["zinc_low_mg"]),

        "source": "nutrition_recommendation_vn.pdf"
    })

In [94]:
trace_file = os.path.join(
    PROCESSED_DIR,
    "recommendation_trace_minerals.json"
)

with open(
    trace_file,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        trace_records_clean,
        f,
        ensure_ascii=False,
        indent=2
    )

print(trace_file)
print("Records:", len(trace_records_clean))

/content/drive/MyDrive/nutrition_rag/data_processed/recommendation_trace_minerals.json
Records: 15


In [97]:
vitamin_table = pdf.pages[5].extract_tables()[0]

print("Rows:", len(vitamin_table))

for row in vitamin_table[:20]:
    print(row)

Rows: 25
['Nhóm tuổi, giới', 'A\nmcga', 'D\nmcgc', 'E\nmgd', 'K\nmcg', 'C\nmgb', 'B\n1\nmg', 'B\n2\nmg', 'B\n3\nmg\nNEe', 'B\n6\nmg', 'B\n9\nmcgf', 'B12\nmcg']
['Trẻ em', None, None, None, None, None, None, None, None, None, None, None]
['< 6 tháng', '375', '5', '3', '6', '25', '0,2', '0,3', '2', '0,1', '80', '0,3']
['6-11 tháng', '400', '5', '4', '9', '30', '0,3', '0,4', '4', '0,3', '80', '0,4']
['1-3 tuổi', '400', '5', '5', '13', '30', '0,5', '0,5', '6', '0,5', '160', '0,9']
['4-6 tuổi', '450', '5', '6', '19', '30', '0,6', '0,6', '8', '0,6', '200', '1,2']
['7-9 tuổi', '500', '5', '7', '24', '35', '0,9', '0,9', '12', '1', '300', '1,8']
['Nam vị thành niên', None, None, None, None, None, None, None, None, None, None, None]
['10-12 tuổi', '600', '5', '10', '34', '65', '1,2', '1,3', '16', '1,3', '400', '2,4']
['13-15 tuổi', None, None, '12', '50', None, None, None, None, None, None, None]
['16-18 tuổi', None, None, '13', '58', None, None, None, None, None, None, None]
['Nam trưởng thành'

In [98]:
for row in vitamin_table:
    print(row)

['Nhóm tuổi, giới', 'A\nmcga', 'D\nmcgc', 'E\nmgd', 'K\nmcg', 'C\nmgb', 'B\n1\nmg', 'B\n2\nmg', 'B\n3\nmg\nNEe', 'B\n6\nmg', 'B\n9\nmcgf', 'B12\nmcg']
['Trẻ em', None, None, None, None, None, None, None, None, None, None, None]
['< 6 tháng', '375', '5', '3', '6', '25', '0,2', '0,3', '2', '0,1', '80', '0,3']
['6-11 tháng', '400', '5', '4', '9', '30', '0,3', '0,4', '4', '0,3', '80', '0,4']
['1-3 tuổi', '400', '5', '5', '13', '30', '0,5', '0,5', '6', '0,5', '160', '0,9']
['4-6 tuổi', '450', '5', '6', '19', '30', '0,6', '0,6', '8', '0,6', '200', '1,2']
['7-9 tuổi', '500', '5', '7', '24', '35', '0,9', '0,9', '12', '1', '300', '1,8']
['Nam vị thành niên', None, None, None, None, None, None, None, None, None, None, None]
['10-12 tuổi', '600', '5', '10', '34', '65', '1,2', '1,3', '16', '1,3', '400', '2,4']
['13-15 tuổi', None, None, '12', '50', None, None, None, None, None, None, None]
['16-18 tuổi', None, None, '13', '58', None, None, None, None, None, None, None]
['Nam trưởng thành', None, N

In [99]:
def is_group_row(row):

    values = row[1:]

    return all(
        v is None or str(v).strip() == ""
        for v in values
    )

vitamin_records = []

current_group = None
last_record = None

for row in vitamin_table[1:]:

    first_col = (
        str(row[0]).strip()
        if row[0]
        else ""
    )

    if not first_col:
        continue

    if is_group_row(row):

        current_group = first_col
        last_record = None
        continue

    record = {
        "group": current_group,
        "age": first_col,

        "vitamin_a_mcg": row[1],
        "vitamin_d_mcg": row[2],
        "vitamin_e_mg": row[3],
        "vitamin_k_mcg": row[4],
        "vitamin_c_mg": row[5],

        "vitamin_b1_mg": row[6],
        "vitamin_b2_mg": row[7],
        "vitamin_b3_mg": row[8],
        "vitamin_b6_mg": row[9],

        "folate_mcg": row[10],
        "vitamin_b12_mcg": row[11]
    }

    if last_record:

        for key in record:

            if key in ["group", "age"]:
                continue

            value = record[key]

            if (
                value is None
                or str(value).strip() == ""
            ):
                record[key] = last_record[key]

    vitamin_records.append(record)

    last_record = record.copy()

print("Records:", len(vitamin_records))

for r in vitamin_records[:15]:
    print(r)

Records: 19
{'group': 'Trẻ em', 'age': '< 6 tháng', 'vitamin_a_mcg': '375', 'vitamin_d_mcg': '5', 'vitamin_e_mg': '3', 'vitamin_k_mcg': '6', 'vitamin_c_mg': '25', 'vitamin_b1_mg': '0,2', 'vitamin_b2_mg': '0,3', 'vitamin_b3_mg': '2', 'vitamin_b6_mg': '0,1', 'folate_mcg': '80', 'vitamin_b12_mcg': '0,3'}
{'group': 'Trẻ em', 'age': '6-11 tháng', 'vitamin_a_mcg': '400', 'vitamin_d_mcg': '5', 'vitamin_e_mg': '4', 'vitamin_k_mcg': '9', 'vitamin_c_mg': '30', 'vitamin_b1_mg': '0,3', 'vitamin_b2_mg': '0,4', 'vitamin_b3_mg': '4', 'vitamin_b6_mg': '0,3', 'folate_mcg': '80', 'vitamin_b12_mcg': '0,4'}
{'group': 'Trẻ em', 'age': '1-3 tuổi', 'vitamin_a_mcg': '400', 'vitamin_d_mcg': '5', 'vitamin_e_mg': '5', 'vitamin_k_mcg': '13', 'vitamin_c_mg': '30', 'vitamin_b1_mg': '0,5', 'vitamin_b2_mg': '0,5', 'vitamin_b3_mg': '6', 'vitamin_b6_mg': '0,5', 'folate_mcg': '160', 'vitamin_b12_mcg': '0,9'}
{'group': 'Trẻ em', 'age': '4-6 tuổi', 'vitamin_a_mcg': '450', 'vitamin_d_mcg': '5', 'vitamin_e_mg': '6', 'vitami

In [101]:
def parse_number(value):

    if value is None:
        return None

    value = str(value).strip()

    if not value:
        return None

    value = value.replace(".", "")
    value = value.replace(",", ".")

    try:
        return float(value)
    except:
        return value

vitamin_records_clean = []

for record in vitamin_records:

    vitamin_records_clean.append({
        "record_type": "vitamin",

        "group": record["group"],
        "age": record["age"],

        "vitamin_a_mcg":
            parse_number(record["vitamin_a_mcg"]),

        "vitamin_d_mcg":
            parse_number(record["vitamin_d_mcg"]),

        "vitamin_e_mg":
            parse_number(record["vitamin_e_mg"]),

        "vitamin_k_mcg":
            parse_number(record["vitamin_k_mcg"]),

        "vitamin_c_mg":
            parse_number(record["vitamin_c_mg"]),

        "vitamin_b1_mg":
            parse_number(record["vitamin_b1_mg"]),

        "vitamin_b2_mg":
            parse_number(record["vitamin_b2_mg"]),

        "vitamin_b3_mg":
            parse_number(record["vitamin_b3_mg"]),

        "vitamin_b6_mg":
            parse_number(record["vitamin_b6_mg"]),

        "folate_mcg":
            parse_number(record["folate_mcg"]),

        "vitamin_b12_mcg":
            parse_number(record["vitamin_b12_mcg"]),

        "source":
            "nutrition_recommendation_vn.pdf"
    })

print(len(vitamin_records_clean))

for r in vitamin_records_clean[:5]:
    print(r)

19
{'record_type': 'vitamin', 'group': 'Trẻ em', 'age': '< 6 tháng', 'vitamin_a_mcg': 375.0, 'vitamin_d_mcg': 5.0, 'vitamin_e_mg': 3.0, 'vitamin_k_mcg': 6.0, 'vitamin_c_mg': 25.0, 'vitamin_b1_mg': 0.2, 'vitamin_b2_mg': 0.3, 'vitamin_b3_mg': 2.0, 'vitamin_b6_mg': 0.1, 'folate_mcg': 80.0, 'vitamin_b12_mcg': 0.3, 'source': 'nutrition_recommendation_vn.pdf'}
{'record_type': 'vitamin', 'group': 'Trẻ em', 'age': '6-11 tháng', 'vitamin_a_mcg': 400.0, 'vitamin_d_mcg': 5.0, 'vitamin_e_mg': 4.0, 'vitamin_k_mcg': 9.0, 'vitamin_c_mg': 30.0, 'vitamin_b1_mg': 0.3, 'vitamin_b2_mg': 0.4, 'vitamin_b3_mg': 4.0, 'vitamin_b6_mg': 0.3, 'folate_mcg': 80.0, 'vitamin_b12_mcg': 0.4, 'source': 'nutrition_recommendation_vn.pdf'}
{'record_type': 'vitamin', 'group': 'Trẻ em', 'age': '1-3 tuổi', 'vitamin_a_mcg': 400.0, 'vitamin_d_mcg': 5.0, 'vitamin_e_mg': 5.0, 'vitamin_k_mcg': 13.0, 'vitamin_c_mg': 30.0, 'vitamin_b1_mg': 0.5, 'vitamin_b2_mg': 0.5, 'vitamin_b3_mg': 6.0, 'vitamin_b6_mg': 0.5, 'folate_mcg': 160.0, 'v

In [105]:
import json
import os

vitamin_output_file = os.path.join(
    PROCESSED_DIR,
    "vitamin_recommendations.json"
)

with open(
    vitamin_output_file,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        vitamin_records_clean,
        f,
        ensure_ascii=False,
        indent=2
    )

print(vitamin_output_file)
print("Records:", len(vitamin_records_clean))

/content/drive/MyDrive/nutrition_rag/data_processed/vitamin_recommendations.json
Records: 19


In [110]:
# ── STEP A + B + C ────────────────────────────────────────────

PREFIX = {
    "food":          "food",
    "mineral":       "min",
    "trace_mineral": "tmn",
    "vitamin":       "vit",
}
counters = {}

def assign_id(rtype):
    p = PREFIX[rtype]
    counters[p] = counters.get(p, 0) + 1
    return f"{p}_{counters[p]:04d}"


def text_food(r):
    lines = [f"Food name: {r.get('food_name_en', 'Unknown')}"]
    if r.get("energy_kcal") is not None: lines.append(f"Calories: {r['energy_kcal']} kcal")
    if r.get("protein_g")   is not None: lines.append(f"Protein: {r['protein_g']} g")
    if r.get("fat_g")       is not None: lines.append(f"Fat: {r['fat_g']} g")
    if r.get("carb_g")      is not None: lines.append(f"Carbohydrate: {r['carb_g']} g")
    if r.get("fiber_g")     is not None: lines.append(f"Fiber: {r['fiber_g']} g")
    if r.get("calcium_mg")  is not None: lines.append(f"Calcium: {r['calcium_mg']} mg")
    if r.get("iron_mg")     is not None: lines.append(f"Iron: {r['iron_mg']} mg")
    return "\n".join(lines)

def text_mineral(r):
    lines = [
        f"Group: {r.get('group', '')}",
        f"Age: {r.get('age', '')}",
    ]
    for field, label, unit in [
        ("calcium_mg",    "Calcium",    "mg"),
        ("magnesium_mg",  "Magnesium",  "mg"),
        ("phosphorus_mg", "Phosphorus", "mg"),
        ("selenium_mcg",  "Selenium",   "mcg"),
    ]:
        if r.get(field) is not None:
            lines.append(f"{label}: {r[field]} {unit}")
    return "\n".join(lines)

def text_trace(r):
    lines = [f"Group: {r.get('group', '')}", f"Age: {r.get('age', '')}"]
    for field, label in [
        ("iodine_mcg",     "Iodine (mcg)"),
        ("iron_5pct_mg",   "Iron 5% bioavailability (mg)"),
        ("iron_10pct_mg",  "Iron 10% bioavailability (mg)"),
        ("iron_15pct_mg",  "Iron 15% bioavailability (mg)"),
        ("zinc_good_mg",   "Zinc high bioavailability (mg)"),
        ("zinc_medium_mg", "Zinc medium bioavailability (mg)"),
        ("zinc_low_mg",    "Zinc low bioavailability (mg)"),
    ]:
        if r.get(field) is not None:
            lines.append(f"{label}: {r[field]}")
    return "\n".join(lines)

def text_vitamin(r):
    lines = [f"Group: {r.get('group', '')}", f"Age: {r.get('age', '')}"]
    for field, label in [
        ("vitamin_a_mcg",   "Vitamin A (mcg)"),
        ("vitamin_d_mcg",   "Vitamin D (mcg)"),
        ("vitamin_e_mg",    "Vitamin E (mg)"),
        ("vitamin_k_mcg",   "Vitamin K (mcg)"),
        ("vitamin_c_mg",    "Vitamin C (mg)"),
        ("vitamin_b1_mg",   "Vitamin B1 (mg)"),
        ("vitamin_b2_mg",   "Vitamin B2 (mg)"),
        ("vitamin_b3_mg",   "Vitamin B3 (mg)"),
        ("vitamin_b6_mg",   "Vitamin B6 (mg)"),
        ("folate_mcg",      "Folate (mcg)"),
        ("vitamin_b12_mcg", "Vitamin B12 (mcg)"),
    ]:
        if r.get(field) is not None:
            lines.append(f"{label}: {r[field]}")
    return "\n".join(lines)

TEXT_FN = {
    "food":          text_food,
    "mineral":       text_mineral,
    "trace_mineral": text_trace,
    "vitamin":       text_vitamin,
}

# food_metadata_clean chưa có record_type → thêm vào
for r in food_metadata_clean:
    r["record_type"] = "food"

all_raw = (
    food_metadata_clean +
    mineral_records +
    trace_records_clean +
    vitamin_records_clean
)

all_records = []
for r in all_raw:
    rtype = r["record_type"]
    all_records.append({
        "record_id":          assign_id(rtype),
        "record_type":        rtype,
        "source":             r.get("source", ""),
        "group":              r.get("group"),
        "age":                r.get("age"),
        "food_name_en":       r.get("food_name_en"),
        "text_for_embedding": TEXT_FN[rtype](r),
    })

print("Total records:", len(all_records))
from collections import Counter
print(Counter(r["record_type"] for r in all_records))

Total records: 583
Counter({'food': 526, 'mineral': 23, 'vitamin': 19, 'trace_mineral': 15})


In [111]:
all_records_file = os.path.join(PROCESSED_DIR, "all_records.json")

with open(all_records_file, "w", encoding="utf-8") as f:
    json.dump(all_records, f, ensure_ascii=False, indent=2)

print("Saved:", all_records_file)
print("Records:", len(all_records))

# Sanity check — xem 1 record mỗi loại
for rtype in ["food", "mineral", "trace_mineral", "vitamin"]:
    sample = next(r for r in all_records if r["record_type"] == rtype)
    print("\n─────", rtype.upper(), "─────")
    print("ID:", sample["record_id"])
    print(sample["text_for_embedding"])

Saved: /content/drive/MyDrive/nutrition_rag/data_processed/all_records.json
Records: 583

───── FOOD ─────
ID: food_0001
Food name: Glutinous rice, milled
Calories: 344.0 kcal
Protein: 8.6 g
Fat: 1.5 g
Carbohydrate: 74.5 g
Fiber: 0.6 g
Calcium: 32.0 mg
Iron: 1.2 mg

───── MINERAL ─────
ID: min_0001
Group: Trẻ em
Age: < 6 tháng
Calcium: 300 mg
Magnesium: 36 mg
Phosphorus: 90 mg
Selenium: 6 mcg

───── TRACE_MINERAL ─────
ID: tmn_0001
Group: Trẻ em
Age: 0-6 tháng
Iodine (mcg): 90.0
Iron 5% bioavailability (mg): 0.93
Zinc high bioavailability (mg): 1.15
Zinc medium bioavailability (mg): 2.86
Zinc low bioavailability (mg): 6.57

───── VITAMIN ─────
ID: vit_0001
Group: Trẻ em
Age: < 6 tháng
Vitamin A (mcg): 375.0
Vitamin D (mcg): 5.0
Vitamin E (mg): 3.0
Vitamin K (mcg): 6.0
Vitamin C (mg): 25.0
Vitamin B1 (mg): 0.2
Vitamin B2 (mg): 0.3
Vitamin B3 (mg): 2.0
Vitamin B6 (mg): 0.1
Folate (mcg): 80.0
Vitamin B12 (mcg): 0.3
